# 02 – N₂ Multi-Solver Benchmark

Demonstrates the **benchmark mode**: same fragment Hamiltonian, three solvers,
side-by-side energy comparison.

```
PySCF HF/STO-3G  →  SimpleCAS (6e, 6o)  →  JW mapper
                  ├── NumPySolver  (FCI reference)
                  ├── VQE-UCCSD
                  └── ADAPT-VQE
```

We also scan the N≡N bond length to show how correlation error grows
at dissociation.

In [ ]:
import os, sys, pathlib
_here = pathlib.Path().resolve()
for _root in [_here, _here.parent, _here.parent.parent]:
    if (_root / "pyproject.toml").is_file() and (_root / "dft_qc_pipeline").is_dir():
        sys.path.insert(0, str(_root))
        break

import logging
logging.basicConfig(level=logging.WARNING, format='%(levelname)s: %(message)s')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_NB = pathlib.Path().resolve()
_CFG_DIR = _NB.parent / "configs"

from dft_qc_pipeline import Pipeline, PipelineConfig
from dft_qc_pipeline.hamiltonian import build_mapper
from dft_qc_pipeline.postprocessing.benchmark import BenchmarkCollector


## 1. Single-point benchmark at N₂ equilibrium

In [ ]:
from dft_qc_pipeline.core.config import MapperConfig

config = PipelineConfig.from_yaml(_CFG_DIR / "n2_compare.yaml")

# Use benchmark mode
config.benchmark_mode = True
if os.environ.get("CI_FAST_NB", "").strip():
    config.benchmark_solvers = ["numpy"]  # CI: NumPy only
else:
    config.benchmark_solvers = ["numpy", "vqe"]
config.solver.type = 'numpy'   # primary solver = NumPy FCI

result = Pipeline(config).run()

frag = result.fragment_results['N2_active']
print(f'FCI energy (NumPy): {frag.energy:.10f} Ha')
if 'benchmark' in frag.extra:
    for sname, bdata in frag.extra['benchmark'].items():
        err_mHa = (bdata['energy'] - frag.energy) * 1000
        print(f'  {sname:<20} E={bdata["energy"]:.10f} Ha  ΔE={err_mHa:.4f} mHa')


## 2. Direct BenchmarkCollector usage

Build the embedded Hamiltonian once, then call the collector explicitly –
useful when you want to iterate over solver hyperparameters.

In [ ]:
from dft_qc_pipeline.classical_backends.pyscf_backend import PySCFBackend
from dft_qc_pipeline.embedding.simple_cas import SimpleCASEmbedding
from dft_qc_pipeline.hamiltonian.fragment_region import FragmentRegion
from dft_qc_pipeline.hamiltonian.mappers import build_mapper
from dft_qc_pipeline.core.config import MapperConfig

# Step 1: classical
backend = PySCFBackend(method='hf', verbose=0)
br = backend.run('N 0 0 0; N 0 0 1.098', 'sto-3g')

# Step 2: embedding
region = FragmentRegion(
    name='N2',
    atom_indices=[0, 1],
    nelec=6,
    norb=6,
    localization='none',
)
emb = SimpleCASEmbedding()
emb_H = emb.embed(br, region)
print(f'Fragment norb={emb_H.norb}, nelec={emb_H.nelec}, e_core={emb_H.e_core:.6f}')

# Step 3: benchmark all solvers
mapper = build_mapper(MapperConfig(type='jw'))
_bc_names = (
    ["numpy"]
    if os.environ.get("CI_FAST_NB", "").strip()
    else ["numpy", "vqe"]
)
bc = BenchmarkCollector(
    solver_names=_bc_names,
    solver_kwargs={'vqe': {'ansatz': 'uccsd', 'optimizer': 'slsqp', 'max_iter': 200}},
)
bench_result = bc.run(emb_H, mapper)
print(bench_result.summary_table())


## 3. Bond dissociation scan: FCI vs HF vs VQE

In [ ]:
if os.environ.get("CI_FAST_NB", "").strip():
    distances = np.linspace(1.0, 2.2, 5)
else:
    distances = np.linspace(0.9, 2.5, 14)
e_hf, e_fci = [], []

for d in distances:
    cfg = PipelineConfig.from_yaml(_CFG_DIR / "n2_compare.yaml")
    cfg.backend.geometry = f'N 0 0 0; N 0 0 {d:.4f}'
    cfg.solver.type = 'numpy'
    res = Pipeline(cfg).run()
    e_hf.append(res.backend_result.energy_hf)
    e_fci.append(res.total_energy)
    print(f'd={d:.2f}  HF={e_hf[-1]:.6f}  FCI={e_fci[-1]:.6f}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.plot(distances, e_hf, 'b--o', ms=4, label='HF')
ax.plot(distances, e_fci, 'r-o', ms=4, label='FCI (CAS 6e,6o)')
ax.set_xlabel('N–N distance (Å)')
ax.set_ylabel('Energy (Ha)')
ax.set_title('N₂ bond dissociation – STO-3G')
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
corr = (np.array(e_fci) - np.array(e_hf)) * 1000
ax.plot(distances, corr, 'g-^', ms=5)
ax.set_xlabel('N–N distance (Å)')
ax.set_ylabel('Correlation energy (mHa)')
ax.set_title('Correlation energy vs bond length')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('N2_dissociation_benchmark.png', dpi=150)
plt.show()


## 4. HEA vs UCCSD at equilibrium

Compare hardware-efficient ansatz and UCCSD on the equilibrium geometry.

In [ ]:
results_table = []

if os.environ.get("CI_FAST_NB", "").strip():
    _ansatz_list = ["uccsd"]
    _vqe_max_iter = 50
else:
    _ansatz_list = ["uccsd", "hea"]
    _vqe_max_iter = 300
for ansatz_name in _ansatz_list:
    cfg = PipelineConfig.from_yaml(_CFG_DIR / "n2_compare.yaml")
    cfg.solver.type = 'vqe'
    cfg.solver.ansatz = ansatz_name
    cfg.solver.optimizer = 'cobyla'
    cfg.solver.max_iter = _vqe_max_iter
    try:
        r = Pipeline(cfg).run()
        results_table.append({'ansatz': ansatz_name, 'energy': r.total_energy})
    except Exception as e:
        results_table.append({'ansatz': ansatz_name, 'energy': float('nan'), 'error': str(e)})

df = pd.DataFrame(results_table)
print(df.to_string(index=False))
